# Chat Interface

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv('SARVAM_API_KEY')

## Tool Callings

In [3]:
from sarvamai import SarvamAI
import json

In [4]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

In [5]:
system_prompt = """
You are a helpful Customer Services Rep working for ABC Bank.
"""

In [6]:
model="sarvam-105b-conversations",


In [7]:
def get_balance(account_number: str):
    if account_number == '001002':
        return "INR 51203"
    else:
        return "INR 2500"

def get_transactions(account_number: str):
    if account_number == "001002":
        return [
            ("Time", "Transaction", "INR"),
            ("11:21 AM", "Debit - XYZ Super Market", "INR 651.50"),
            ("4:45 PM",  "Credit - Refund from PQR Braodband Services", "INR 999"),
        ]
    else:
        return "No transactions in the last 24 hrs in your account"

In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
     {
        "type": "function",
        "function": {
            "name": "get_transactions",
            "description": "Get transactions from the last 24 hours",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },   
]

In [9]:
tools_map  = {
    "get_balance": get_balance,
    "get_transactions": get_transactions,
}

def invoke_tool(f_name: str, f_args: dict):
    return tools_map[f_name](**f_args)

In [10]:
def handle_tool_call(tool_call: any, messagaes: list):
    f_name = tool_call.function.name
    f_args = json.loads(tool_call.function.arguments)
    result = invoke_tool(f_name, f_args)
    
    messages.append(
        {
            "role": "assistant",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
            ],
        }
    )
    
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        }
    ) 

In [11]:
messages=[
    {"role": "system", "content": system_prompt},
]

In [12]:
def chat(user_message: str, *args):
    global messages

    messages.append(
        {"role": "user", "content": user_message.strip()}
    )

    while True:
        response = client.chat.completions(
            model="sarvam-105b-conversations",
            messages=messages,
            tools=tools,
        )
    
        if response.choices[0].message.tool_calls:
            message = response.choices[0].message
            for tool_call in message.tool_calls:
                handle_tool_call(tool_call, messages)
            continue
        break

    assistant_response = response.choices[0].message.content.strip()
    # print('\nAssistant: ', assistant_response)
    messages.append(
        {"role": "assistant", "content": assistant_response}
    )
    return assistant_response

## Setup Gradio

In [13]:
import gradio as gr

chat_interface = gr.ChatInterface(fn=chat, title="ABC Bank - Customer Service Chat")
chat_interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [14]:
chat_interface.close()

Closing server running on port: 7860
